In [2]:
!pip install --upgrade geopandas
!pip install --upgrade pyshp

!pip install --upgrade shapely

!pip install --upgrade descartes
!pip install plotly_express

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 1.1 MB 5.2 MB/s 
     |████████████████████████████████| 16.6 MB 135 kB/s 
     |████████████████████████████████| 7.8 MB 50.0 MB/s 
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 46 kB 2.2 MB/s 
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [3]:
import geopandas as gpd
import pandas as pd
import shapely 
import numpy as np
from shapely.geometry import Point
import matplotlib
import matplotlib.pyplot as plt 
import folium
import plotly_express as px
import json
import pandas as pd

In [4]:
def getndvi(file):
  ndvi = pd.read_csv(file)
  ndvi_reduced = ndvi[['latitude',	'longitude',	'mean',	'name']]
  ndvi_reduced = ndvi_reduced.rename(columns={"mean": "ndvi"})
  return ndvi_reduced

def getevi(file_evi):
  evi = pd.read_csv(file_evi)
  evi_reduced = evi[['latitude',	'longitude',	'mean',	'name']]
  evi_reduced = evi_reduced.rename(columns={"mean": "evi"})
  ev = evi_reduced.evi
  return ev

def getsavi(file_savi):
  savi = pd.read_csv(file_savi)
  savi_reduced = savi[['latitude',	'longitude',	'mean',	'name']]
  savi_reduced = savi_reduced.rename(columns={"mean": "savi"})
  sav = savi_reduced.savi
  return sav

def getgndvi(file_gndvi):
  gndvi = pd.read_csv(file_gndvi)
  gndvi_reduced = gndvi[['latitude',	'longitude',	'mean',	'name']]
  gndvi_reduced = gndvi_reduced.rename(columns={"mean": "gndvi"})
  gndvi = gndvi_reduced.gndvi
  return gndvi

def joined(ndvi, evi,savi,gndvi):
  ndvi_reduced = getndvi(ndvi)
  ev = getevi(evi)
  sav = getsavi(savi)
  gndvi = getgndvi(gndvi)
  
  ndvi_reduced.insert(3, 'evi', ev)
  c = ndvi_reduced

  c.insert(3, 'savi', sav)
  b= c
  b.insert(3,'gndvi', gndvi)

  return b

In [5]:
def getdata(ndvi,evi,savi, gndvi, fielddata, year):
  crop = pd.read_csv(fielddata)
  combined = joined(ndvi,evi,savi, gndvi)
  combined['name'] = combined['name'].str.upper()
  combined_sorted = combined.sort_values("name")
  combined_sorted_r = combined_sorted.reset_index(drop= 'TRUE')
 
  cornYear = crop[(crop['Year'] == year)]
  cornYear_counties = cornYear[(cornYear.County != 'OTHER COUNTIES')]
  cornarea = cornYear_counties[cornYear_counties['Data Item'] == "CORN - ACRES PLANTED"]
  cornYield = cornYear_counties[cornYear_counties['Data Item'] == "CORN, GRAIN - YIELD, MEASURED IN BU / ACRE"]
  
  temp_corn_yield = cornYield[['County','Value']]
  temp_corn_yield =  temp_corn_yield.rename(columns={"Value": "Yield_Bu_Acr"})
  temp_corn_area = cornarea[['County','Value']]
  corn_yield_area = temp_corn_yield.merge(temp_corn_area[['County', 'Value']], left_on='County', right_on='County', how='inner' )
  corn_yield_area = corn_yield_area.rename(columns={"Value": "Area_planted_Acr"}) 
  # temp_corn_area.insert(1, 'Yield_Bu/Ac', [temp_corn_yield.Yield_Bu_Acr])
  # joined_corn = temp_corn_area
  data_year = combined_sorted_r.merge(corn_yield_area[['County', 'Yield_Bu_Acr','Area_planted_Acr']], left_on='name', right_on='County', how='inner' )
  data_year= data_year.dropna()
  data_corn_year = data_year.drop('name', axis =1)
  
  # corn_dataofYear = data_year.rename(columns={"Value": "Area_planted"})
  # data_year['Value'] = data_year['Value'].fillna('Nan')
  # data_year.dropna()
  data_corn_year['Year']= year

  return  data_corn_year

In [6]:
ndvi = '/content/drive/MyDrive/geotiff/file19/NDVI_mean_19.csv'
evi = '/content/drive/MyDrive/geotiff/file19/evi_mean_19.csv'
savi ='/content/drive/MyDrive/geotiff/file19/SAVI_mean_19.csv'
gndvi ='/content/drive/MyDrive/geotiff/file19/GNDVI_mean_19.csv'
fielddata = '/content/drive/MyDrive/geotiff/cropData2013-2021_YIELD_AREA.csv'

In [7]:
data_corn_19 = getdata(ndvi,evi,savi, gndvi, fielddata, 2019)
data_corn_19

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.848317,7.503676,1.274267,24.595310,ADAMS,170.1,"40,000",2019
1,41.091863,-85.072235,0.733248,5.745552,1.101435,30.529547,ALLEN,181.4,"53,000",2019
2,40.608260,-87.315485,0.880978,9.454374,1.321387,2.677317,BENTON,186,"123,000",2019
3,40.050899,-86.469019,0.813557,7.233196,1.220665,7.096993,BOONE,177.4,"72,000",2019
4,40.584988,-86.565147,0.884565,9.792332,1.327004,13.060968,CARROLL,194.6,"93,000",2019
...,...,...,...,...,...,...,...,...,...,...
63,38.089770,-87.298307,0.849386,7.886705,1.281135,29.812472,WARRICK,156.5,"36,000",2019
64,38.600620,-86.104756,0.746633,5.453854,1.120373,15.149207,WASHINGTON,134.3,"45,000",2019
65,39.863098,-85.006740,0.891594,8.966701,1.347580,17.620357,WAYNE,167.5,"61,000",2019
66,40.748952,-86.888849,0.876617,9.316898,1.314930,7.654961,WHITE,177.6,"135,000",2019


In [8]:
ndvi = '/content/drive/MyDrive/geotiff/file18/NDVI_mean_18.csv'
evi = '/content/drive/MyDrive/geotiff/file18/evi_mean_18.csv'
savi ='/content/drive/MyDrive/geotiff/file18/SAVI_mean_18.csv'
gndvi ='/content/drive/MyDrive/geotiff/file18/GNDVI_mean_18.csv'
fielddata = '/content/drive/MyDrive/geotiff/cropData2013-2021_YIELD_AREA.csv'

In [9]:
data_corn_18 = getdata(ndvi,evi,savi, gndvi, fielddata, 2018)
data_corn_18

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.893652,9.889491,1.341166,12.438407,ADAMS,194.5,"58,000",2018
1,41.091863,-85.072235,0.884231,9.389818,1.326631,8.760394,ALLEN,181.7,"74,000",2018
2,39.206483,-85.893663,0.875999,9.792190,1.314939,7.100656,BARTHOLOMEW,189.3,"55,000",2018
3,40.608260,-87.315485,0.897025,10.678445,1.345771,5.099712,BENTON,201.7,"118,000",2018
4,40.474668,-85.326241,0.887061,9.801643,1.330516,9.116563,BLACKFORD,201,"29,000",2018
...,...,...,...,...,...,...,...,...,...,...
74,38.600620,-86.104756,0.896007,11.011435,1.343924,9.617093,WASHINGTON,184.1,"37,200",2018
75,39.863098,-85.006740,0.886975,9.389760,1.330730,41.895976,WAYNE,191.7,"62,000",2018
76,40.749954,-85.212428,0.894653,10.129063,1.342205,7.160000,WELLS,204.1,"75,000",2018
77,40.748952,-86.888849,0.892409,10.241195,1.338905,9.045582,WHITE,196.9,"135,000",2018


In [10]:
ndvi = '/content/drive/MyDrive/geotiff/file17/NDVI_mean_17.csv'
evi = '/content/drive/MyDrive/geotiff/file17/evi_mean_17.csv'
savi ='/content/drive/MyDrive/geotiff/file17/SAVI_mean_17.csv'
gndvi ='/content/drive/MyDrive/geotiff/file17/GNDVI_mean_17.csv'
fielddata = '/content/drive/MyDrive/geotiff/cropData2013-2021_YIELD_AREA.csv'

data_corn_17 = getdata(ndvi,evi,savi, gndvi, fielddata, 2017)
data_corn_17

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.864625,8.241853,1.297499,30.436149,ADAMS,169.8,"56,000",2017
1,41.091863,-85.072235,0.847565,7.749038,1.271482,19.049234,ALLEN,167.6,"70,000",2017
2,40.608260,-87.315485,0.745131,5.575977,1.118520,30.377186,BENTON,199.2,"123,000",2017
3,40.474668,-85.326241,0.871348,8.705602,1.307127,9.052742,BLACKFORD,179.2,"27,000",2017
4,40.050899,-86.469019,0.878647,8.803680,1.318123,8.285679,BOONE,184.4,"90,000",2017
...,...,...,...,...,...,...,...,...,...,...
79,38.600620,-86.104756,0.886354,9.983007,1.329634,5.891504,WASHINGTON,172.6,"39,000",2017
80,39.863098,-85.006740,0.891294,9.310294,1.339089,17.152963,WAYNE,171.9,"62,000",2017
81,40.749954,-85.212428,0.874000,8.797505,1.311573,14.264370,WELLS,185.1,"76,000",2017
82,40.748952,-86.888849,0.748240,5.425997,1.122379,18.759820,WHITE,185.4,"139,000",2017


In [11]:
ndvi = '/content/drive/MyDrive/geotiff/file16/NDVI_mean_16.csv'
evi = '/content/drive/MyDrive/geotiff/file16/evi_mean_16.csv'
savi ='/content/drive/MyDrive/geotiff/file16/SAVI_mean_16.csv'
gndvi ='/content/drive/MyDrive/geotiff/file16/GNDVI_mean_16.csv'
fielddata = '/content/drive/MyDrive/geotiff/cropData2013-2021_YIELD_AREA.csv'

data_corn_16 = getdata(ndvi,evi,savi, gndvi, fielddata, 2016)
data_corn_16

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.847294,8.059073,1.273994,30.930936,ADAMS,159.5,"61,000",2016
1,41.091863,-85.072235,0.809516,7.284379,1.215140,12.846213,ALLEN,157.9,"78,000",2016
2,39.206483,-85.893663,0.579251,3.582237,0.871630,7.109463,BARTHOLOMEW,171.2,"60,000",2016
3,40.608260,-87.315485,0.887275,9.919268,1.330856,4.196291,BENTON,199.2,"125,000",2016
4,40.474668,-85.326241,0.782445,6.237901,1.174111,21.862807,BLACKFORD,173.9,"30,000",2016
...,...,...,...,...,...,...,...,...,...,...
75,38.089770,-87.298307,0.870785,8.786237,1.306328,5.728735,WARRICK,134.6,"32,000",2016
76,38.600620,-86.104756,0.663781,4.250273,0.995599,2.024451,WASHINGTON,147.8,"43,500",2016
77,39.863098,-85.006740,0.876016,9.249828,1.315147,8.654324,WAYNE,167.2,"64,000",2016
78,40.748952,-86.888849,0.895148,10.283811,1.343027,4.374058,WHITE,189.2,"141,000",2016


In [12]:
ndvi = '/content/drive/MyDrive/geotiff/file15/NDVI_mean_15.csv'
evi = '/content/drive/MyDrive/geotiff/file15/evi_mean_15.csv'
savi ='/content/drive/MyDrive/geotiff/file15/SAVI_mean_15.csv'
gndvi ='/content/drive/MyDrive/geotiff/file15/GNDVI_mean_15.csv'
fielddata = '/content/drive/MyDrive/geotiff/cropData2013-2021_YIELD_AREA.csv'

data_corn_15 = getdata(ndvi,evi,savi, gndvi, fielddata, 2015)
data_corn_15

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.777858,5.232603,1.166678,15.985957,ADAMS,103.5,"61,000",2015
1,41.091863,-85.072235,0.810941,6.356497,1.216393,31.696300,ALLEN,120.7,"74,000",2015
2,39.206483,-85.893663,0.731020,4.977475,1.096432,24.299013,BARTHOLOMEW,161,"60,000",2015
3,40.608260,-87.315485,0.858471,7.747026,1.288887,27.139705,BENTON,169.3,"129,000",2015
4,40.474668,-85.326241,0.809500,6.143916,1.214161,2.422544,BLACKFORD,117.5,"29,000",2015
...,...,...,...,...,...,...,...,...,...,...
79,38.600620,-86.104756,0.859901,8.365874,1.289789,10.858726,WASHINGTON,156,"49,000",2015
80,39.863098,-85.006740,0.860092,7.765846,1.290085,12.129338,WAYNE,161.2,"66,000",2015
81,40.749954,-85.212428,0.757116,5.145349,1.135569,4.361292,WELLS,112.5,"78,000",2015
82,40.748952,-86.888849,0.834210,6.326863,1.255143,11.789814,WHITE,128,"144,000",2015


In [13]:
ndvi = '/content/drive/MyDrive/geotiff/file14/NDVI_mean_14.csv'
evi = '/content/drive/MyDrive/geotiff/file14/evi_mean_14.csv'
savi ='/content/drive/MyDrive/geotiff/file14/SAVI_mean_14.csv'
gndvi ='/content/drive/MyDrive/geotiff/file14/GNDVI_mean_14.csv'
fielddata = '/content/drive/MyDrive/geotiff/cropData2013-2021_YIELD_AREA.csv'

data_corn_14 = getdata(ndvi,evi,savi, gndvi, fielddata, 2014)
data_corn_14

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.864546,8.262665,1.296773,18.822446,ADAMS,183.3,"64,000",2014
1,41.091863,-85.072235,0.849503,8.167988,1.274335,22.839722,ALLEN,185.8,"82,000",2014
2,39.206483,-85.893663,0.608369,3.347106,0.912521,12.452923,BARTHOLOMEW,200.3,"60,000",2014
3,40.608260,-87.315485,0.898764,9.688027,1.348978,6.348873,BENTON,202.4,"130,000",2014
4,40.474668,-85.326241,0.677666,4.744491,1.016690,14.957222,BLACKFORD,157.1,"32,500",2014
...,...,...,...,...,...,...,...,...,...,...
81,38.600620,-86.104756,0.695545,4.953170,1.051620,3.848346,WASHINGTON,180,"49,000",2014
82,39.863098,-85.006740,0.843424,7.593557,1.265623,15.691030,WAYNE,185.2,"65,000",2014
83,40.749954,-85.212428,0.772672,6.416721,1.159055,13.027123,WELLS,189.9,"81,000",2014
84,40.748952,-86.888849,0.894939,9.616665,1.342663,10.291662,WHITE,194.2,"155,000",2014


In [14]:
data_corn_14_15 = data_corn_14.append(data_corn_15, ignore_index=True)
data_corn_14_15

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.864546,8.262665,1.296773,18.822446,ADAMS,183.3,"64,000",2014
1,41.091863,-85.072235,0.849503,8.167988,1.274335,22.839722,ALLEN,185.8,"82,000",2014
2,39.206483,-85.893663,0.608369,3.347106,0.912521,12.452923,BARTHOLOMEW,200.3,"60,000",2014
3,40.608260,-87.315485,0.898764,9.688027,1.348978,6.348873,BENTON,202.4,"130,000",2014
4,40.474668,-85.326241,0.677666,4.744491,1.016690,14.957222,BLACKFORD,157.1,"32,500",2014
...,...,...,...,...,...,...,...,...,...,...
165,38.600620,-86.104756,0.859901,8.365874,1.289789,10.858726,WASHINGTON,156,"49,000",2015
166,39.863098,-85.006740,0.860092,7.765846,1.290085,12.129338,WAYNE,161.2,"66,000",2015
167,40.749954,-85.212428,0.757116,5.145349,1.135569,4.361292,WELLS,112.5,"78,000",2015
168,40.748952,-86.888849,0.834210,6.326863,1.255143,11.789814,WHITE,128,"144,000",2015


In [15]:
data_corn_14_15_16 = data_corn_14_15.append(data_corn_16, ignore_index=True)
data_corn_14_15_16

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.864546,8.262665,1.296773,18.822446,ADAMS,183.3,"64,000",2014
1,41.091863,-85.072235,0.849503,8.167988,1.274335,22.839722,ALLEN,185.8,"82,000",2014
2,39.206483,-85.893663,0.608369,3.347106,0.912521,12.452923,BARTHOLOMEW,200.3,"60,000",2014
3,40.608260,-87.315485,0.898764,9.688027,1.348978,6.348873,BENTON,202.4,"130,000",2014
4,40.474668,-85.326241,0.677666,4.744491,1.016690,14.957222,BLACKFORD,157.1,"32,500",2014
...,...,...,...,...,...,...,...,...,...,...
245,38.089770,-87.298307,0.870785,8.786237,1.306328,5.728735,WARRICK,134.6,"32,000",2016
246,38.600620,-86.104756,0.663781,4.250273,0.995599,2.024451,WASHINGTON,147.8,"43,500",2016
247,39.863098,-85.006740,0.876016,9.249828,1.315147,8.654324,WAYNE,167.2,"64,000",2016
248,40.748952,-86.888849,0.895148,10.283811,1.343027,4.374058,WHITE,189.2,"141,000",2016


In [16]:
data_corn_14_15_16_17 = data_corn_14_15_16.append(data_corn_17, ignore_index=True)
data_corn_14_15_16_17

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.864546,8.262665,1.296773,18.822446,ADAMS,183.3,"64,000",2014
1,41.091863,-85.072235,0.849503,8.167988,1.274335,22.839722,ALLEN,185.8,"82,000",2014
2,39.206483,-85.893663,0.608369,3.347106,0.912521,12.452923,BARTHOLOMEW,200.3,"60,000",2014
3,40.608260,-87.315485,0.898764,9.688027,1.348978,6.348873,BENTON,202.4,"130,000",2014
4,40.474668,-85.326241,0.677666,4.744491,1.016690,14.957222,BLACKFORD,157.1,"32,500",2014
...,...,...,...,...,...,...,...,...,...,...
329,38.600620,-86.104756,0.886354,9.983007,1.329634,5.891504,WASHINGTON,172.6,"39,000",2017
330,39.863098,-85.006740,0.891294,9.310294,1.339089,17.152963,WAYNE,171.9,"62,000",2017
331,40.749954,-85.212428,0.874000,8.797505,1.311573,14.264370,WELLS,185.1,"76,000",2017
332,40.748952,-86.888849,0.748240,5.425997,1.122379,18.759820,WHITE,185.4,"139,000",2017


In [17]:
data_corn_14_15_16_17_18 = data_corn_14_15_16_17.append(data_corn_18, ignore_index=True)
data_corn_14_15_16_17_18

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.864546,8.262665,1.296773,18.822446,ADAMS,183.3,"64,000",2014
1,41.091863,-85.072235,0.849503,8.167988,1.274335,22.839722,ALLEN,185.8,"82,000",2014
2,39.206483,-85.893663,0.608369,3.347106,0.912521,12.452923,BARTHOLOMEW,200.3,"60,000",2014
3,40.608260,-87.315485,0.898764,9.688027,1.348978,6.348873,BENTON,202.4,"130,000",2014
4,40.474668,-85.326241,0.677666,4.744491,1.016690,14.957222,BLACKFORD,157.1,"32,500",2014
...,...,...,...,...,...,...,...,...,...,...
408,38.600620,-86.104756,0.896007,11.011435,1.343924,9.617093,WASHINGTON,184.1,"37,200",2018
409,39.863098,-85.006740,0.886975,9.389760,1.330730,41.895976,WAYNE,191.7,"62,000",2018
410,40.749954,-85.212428,0.894653,10.129063,1.342205,7.160000,WELLS,204.1,"75,000",2018
411,40.748952,-86.888849,0.892409,10.241195,1.338905,9.045582,WHITE,196.9,"135,000",2018


In [18]:
data_corn_14_15_16_17_18_19 = data_corn_14_15_16_17_18.append(data_corn_19, ignore_index=True)
data_corn_14_15_16_17_18_19

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year
0,40.745740,-84.936135,0.864546,8.262665,1.296773,18.822446,ADAMS,183.3,"64,000",2014
1,41.091863,-85.072235,0.849503,8.167988,1.274335,22.839722,ALLEN,185.8,"82,000",2014
2,39.206483,-85.893663,0.608369,3.347106,0.912521,12.452923,BARTHOLOMEW,200.3,"60,000",2014
3,40.608260,-87.315485,0.898764,9.688027,1.348978,6.348873,BENTON,202.4,"130,000",2014
4,40.474668,-85.326241,0.677666,4.744491,1.016690,14.957222,BLACKFORD,157.1,"32,500",2014
...,...,...,...,...,...,...,...,...,...,...
476,38.089770,-87.298307,0.849386,7.886705,1.281135,29.812472,WARRICK,156.5,"36,000",2019
477,38.600620,-86.104756,0.746633,5.453854,1.120373,15.149207,WASHINGTON,134.3,"45,000",2019
478,39.863098,-85.006740,0.891594,8.966701,1.347580,17.620357,WAYNE,167.5,"61,000",2019
479,40.748952,-86.888849,0.876617,9.316898,1.314930,7.654961,WHITE,177.6,"135,000",2019


In [19]:
Area = data_corn_14_15_16_17_18_19['Area_planted_Acr'].str.replace(',', '').astype(float)
# areas = data_corn_15_16_17_18_19['Area_planted_Acr'].astype(float)
Area

0       64000.0
1       82000.0
2       60000.0
3      130000.0
4       32500.0
         ...   
476     36000.0
477     45000.0
478     61000.0
479    135000.0
480     54000.0
Name: Area_planted_Acr, Length: 481, dtype: float64

In [20]:
yields = data_corn_14_15_16_17_18_19['Yield_Bu_Acr'].astype(float)
yields

0      183.3
1      185.8
2      200.3
3      202.4
4      157.1
       ...  
476    156.5
477    134.3
478    167.5
479    177.6
480    174.0
Name: Yield_Bu_Acr, Length: 481, dtype: float64

In [21]:
data_corn_14_15_16_17_18_19['Area_planted_Acr_f'] = Area


In [22]:
data_corn_14_15_16_17_18_19['yield_Bu_Acr_f'] = yields
data_corn_14_15_16_17_18_19

,latitude,longitude,ndvi,gndvi,savi,evi,County,Yield_Bu_Acr,Area_planted_Acr,Year,Area_planted_Acr_f,yield_Bu_Acr_f
0,40.745740,-84.936135,0.864546,8.262665,1.296773,18.822446,ADAMS,183.3,"64,000",2014,64000.0,183.3
1,41.091863,-85.072235,0.849503,8.167988,1.274335,22.839722,ALLEN,185.8,"82,000",2014,82000.0,185.8
2,39.206483,-85.893663,0.608369,3.347106,0.912521,12.452923,BARTHOLOMEW,200.3,"60,000",2014,60000.0,200.3
3,40.608260,-87.315485,0.898764,9.688027,1.348978,6.348873,BENTON,202.4,"130,000",2014,130000.0,202.4
4,40.474668,-85.326241,0.677666,4.744491,1.016690,14.957222,BLACKFORD,157.1,"32,500",2014,32500.0,157.1
...,...,...,...,...,...,...,...,...,...,...,...,...
476,38.089770,-87.298307,0.849386,7.886705,1.281135,29.812472,WARRICK,156.5,"36,000",2019,36000.0,156.5
477,38.600620,-86.104756,0.746633,5.453854,1.120373,15.149207,WASHINGTON,134.3,"45,000",2019,45000.0,134.3
478,39.863098,-85.006740,0.891594,8.966701,1.347580,17.620357,WAYNE,167.5,"61,000",2019,61000.0,167.5
479,40.748952,-86.888849,0.876617,9.316898,1.314930,7.654961,WHITE,177.6,"135,000",2019,135000.0,177.6


In [23]:
data_corn_14_15_16_17_18_19 = data_corn_14_15_16_17_18_19.drop(['Yield_Bu_Acr','Area_planted_Acr'], axis =1)

In [24]:
data_corn_14_15_16_17_18_19['yield_total_Bu'] = data_corn_14_15_16_17_18_19.Area_planted_Acr_f*data_corn_14_15_16_17_18_19.yield_Bu_Acr_f
data_corn_14_15_16_17_18_19

,latitude,longitude,ndvi,gndvi,savi,evi,County,Year,Area_planted_Acr_f,yield_Bu_Acr_f,yield_total_Bu
0,40.745740,-84.936135,0.864546,8.262665,1.296773,18.822446,ADAMS,2014,64000.0,183.3,11731200.0
1,41.091863,-85.072235,0.849503,8.167988,1.274335,22.839722,ALLEN,2014,82000.0,185.8,15235600.0
2,39.206483,-85.893663,0.608369,3.347106,0.912521,12.452923,BARTHOLOMEW,2014,60000.0,200.3,12018000.0
3,40.608260,-87.315485,0.898764,9.688027,1.348978,6.348873,BENTON,2014,130000.0,202.4,26312000.0
4,40.474668,-85.326241,0.677666,4.744491,1.016690,14.957222,BLACKFORD,2014,32500.0,157.1,5105750.0
...,...,...,...,...,...,...,...,...,...,...,...
476,38.089770,-87.298307,0.849386,7.886705,1.281135,29.812472,WARRICK,2019,36000.0,156.5,5634000.0
477,38.600620,-86.104756,0.746633,5.453854,1.120373,15.149207,WASHINGTON,2019,45000.0,134.3,6043500.0
478,39.863098,-85.006740,0.891594,8.966701,1.347580,17.620357,WAYNE,2019,61000.0,167.5,10217500.0
479,40.748952,-86.888849,0.876617,9.316898,1.314930,7.654961,WHITE,2019,135000.0,177.6,23976000.0


In [25]:
path = '/content/drive/MyDrive/geotiff/dataset_toMakemodel.csv'
with open(path, 'w', encoding = 'utf-8-sig') as f:
  data_corn_14_15_16_17_18_19.to_csv(f)